# Preprocessing TravelFit — 2.337 destinasi
Notebook ini membaca Excel gabungan tervalidasi. **437 baris pihak ketiga** menjadi referensi transformasi K-Means; **1.900 baris sintetis** hanya ditransformasikan untuk simulasi/proyeksi, bukan bukti destinasi nyata. Harga, rating, fasilitas, dan koordinat sintetis belum diverifikasi per lokasi. C3 (jarak), C5 (kategori), dan C6 (hobi) dihitung saat pengguna meminta rekomendasi, bukan di notebook ini.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

FILENAME = 'TravelFit_Dataset_2337_Tervalidasi.xlsx'
LOCAL_DIR = Path.cwd().parent / 'outputs' / 'travelfit_2337' if Path.cwd().name == 'notebooks' else Path.cwd() / 'outputs' / 'travelfit_2337'
candidates = [LOCAL_DIR / FILENAME, Path.cwd() / FILENAME]
try:
    from google.colab import drive
    drive.mount('/content/drive')
    candidates.insert(0, Path('/content/drive/MyDrive/TravelFit') / FILENAME)
except ImportError:
    pass
INPUT = next((path for path in candidates if path.exists()), None)
if INPUT is None:
    raise FileNotFoundError('Excel tidak ditemukan. Salin file ke Drive Saya/TravelFit atau ke outputs/travelfit_2337. Dicari: ' + ', '.join(map(str, candidates)))
OUTPUT_DIR = INPUT.parent / 'preprocessing_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT)
print('Output:', OUTPUT_DIR)

In [ ]:
destinations = pd.read_excel(INPUT, sheet_name='Destinasi_Gabungan')
required = ['place_id', 'place_name', 'city', 'province', 'price', 'rating', 'lat', 'long', 'category_clean', 'c4_facility_score', 'source_dataset', 'record_origin', 'validation_status']
missing = sorted(set(required) - set(destinations.columns))
if missing:
    raise ValueError(f'Kolom wajib tidak ada: {missing}')
expected_sources = {'real_jawa_437': 437, 'synthetic_38_provinsi': 1900}
assert len(destinations) == 2337, f'Jumlah baris bukan 2337: {len(destinations)}'
assert destinations.source_dataset.value_counts().to_dict() == expected_sources
assert destinations.place_id.is_unique, 'place_id harus unik'
print('Baris:', len(destinations))
print(destinations.groupby(['source_dataset', 'record_origin']).size().to_string())

In [ ]:
# Bersihkan tipe dan teks, tetapi jangan mengarang nilai destinasi yang tidak tersedia.
text_cols = ['place_name', 'city', 'province', 'category_clean', 'description', 'activity_tags']
for col in text_cols:
    if col in destinations:
        destinations[col] = destinations[col].astype('string').str.strip()
numeric_cols = ['price', 'rating', 'lat', 'long', 'c4_facility_score', 'c1_ticket_price', 'c2_rating']
for col in numeric_cols:
    if col in destinations:
        destinations[col] = pd.to_numeric(destinations[col], errors='coerce')
essential = ['place_name', 'city', 'province', 'category_clean', 'price', 'rating', 'lat', 'long']
if destinations[essential].isna().any().any():
    raise ValueError('Ada nilai kosong/tidak numerik pada kolom esensial; periksa baris sumber, jangan isi otomatis.')
if ((destinations.price < 0) | (~destinations.rating.between(0, 5)) | (~destinations.lat.between(-11, 7)) | (~destinations.long.between(94, 142))).any():
    raise ValueError('Harga/rating/koordinat di luar batas pemeriksaan dasar.')
key = destinations.place_name.str.casefold() + '|' + destinations.city.str.casefold()
assert not key.duplicated().any(), 'Duplikat nama+city perlu ditinjau manual'
destinations['activity_tags'] = destinations['activity_tags'].fillna('')
destinations['kmeans_role'] = np.where(destinations.source_dataset.eq('real_jawa_437'), 'reference_437', 'projection_only_1900')
print('Provinsi total:', destinations.province.nunique())
print('Kosong time_minutes:', destinations.time_minutes.isna().sum())
print('Kosong activity_tags sesudah isi string kosong:', destinations.activity_tags.isna().sum())

In [ ]:
# Sesuai implementasi website: K-Means memakai harga, rating, dan one-hot kategori.
# Parameter normalisasi untuk proyeksi 2337 baris dipelajari HANYA dari 437 baris sumber.
reference = destinations.loc[destinations.kmeans_role.eq('reference_437')]
features_num = ['price', 'rating']
price_cap = np.percentile(reference.price, 99)
reference_numeric = reference[features_num].copy()
reference_numeric['price'] = reference_numeric.price.clip(upper=price_cap)
full_numeric = destinations[features_num].copy()
full_numeric['price'] = full_numeric.price.clip(upper=price_cap)
mean = reference_numeric.mean()
std = reference_numeric.std(ddof=0).replace(0, 1)
scaled = (full_numeric - mean) / std
scaled.columns = ['price_z', 'rating_z']
categories = sorted(reference.category_clean.dropna().unique())
if not destinations.category_clean.isin(categories).all():
    raise ValueError('Ada kategori sintetis yang tidak muncul dalam 437 baris referensi.')
one_hot = pd.get_dummies(destinations.category_clean, prefix='category', dtype=int)
one_hot = one_hot.reindex(columns=[f'category_{name}' for name in categories], fill_value=0)
kmeans_ready = pd.concat([destinations[['place_id', 'place_name', 'city', 'province', 'source_dataset', 'kmeans_role']].reset_index(drop=True), scaled.reset_index(drop=True), one_hot.reset_index(drop=True)], axis=1)
assert len(kmeans_ready) == 2337 and not kmeans_ready.isna().any().any()
print('K-Means-ready:', kmeans_ready.shape, '| Referensi:', len(reference), '| Proyeksi sintetis:', len(destinations) - len(reference), '| Batas harga p99:', price_cap)
print('Catatan: untuk evaluasi holdout, pecah 437 baris terlebih dahulu, lalu fit mean/std hanya pada train; file ini untuk proyeksi/deployment, bukan pengukuran holdout.')

In [ ]:
clean_path = OUTPUT_DIR / 'destinations_clean_2337.csv'
features_path = OUTPUT_DIR / 'kmeans_ready_2337.csv'
destinations.to_csv(clean_path, index=False, encoding='utf-8-sig')
kmeans_ready.to_csv(features_path, index=False, encoding='utf-8-sig')
assert len(pd.read_csv(clean_path)) == len(pd.read_csv(features_path)) == 2337
print('Tersimpan:', clean_path, features_path, sep='\n- ')